# Practical DSA: Ridge Regression with Cross-Validation (RidgeCV)
**Dataset:** scikit-learn Diabetes (plus a simulated multicollinearity dataset)

**Goals:** baseline OLS vs RidgeCV, alpha tuning with 5-fold CV, coefficient shrinkage, multicollinearity, and an extension section for your own contribution.

Run all cells in order (Runtime > Run all), then save the notebook with outputs (File > Save).

**Student name:** ____________  **Date:** ____________

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Load the Dataset and Understand Its Structure

In [ ]:
dataset = load_diabetes(as_frame=True)
df = dataset.frame.copy()
print("Dataset shape:", df.shape)
display(df.head())

X = df.drop(columns="target")
y = df["target"]
print("\nPredictor columns:")
print(X.columns.tolist())
print("\nTarget summary:")
display(y.describe())

## 3. Descriptive Statistics

In [ ]:
print("Missing values per column:")
display(df.isna().sum())
print("\nDescriptive statistics:")
display(df.describe().T)

summary = pd.DataFrame({
    "mean": X.mean(),
    "median": X.median(),
    "std_dev": X.std(),
    "variance": X.var(),
    "min": X.min(),
    "max": X.max()
})
display(summary.round(3))

## 4. Visualization: Distributions and Correlation

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(y, kde=True)
plt.title("Distribution of the Target Variable")
plt.xlabel("Disease progression measure")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0, annot=False)
plt.title("Feature and Target Correlation Matrix")
plt.show()

target_corr = df.corr(numeric_only=True)["target"].drop("target")
display(target_corr.reindex(
    target_corr.abs().sort_values(ascending=False).index
).to_frame("correlation_with_target"))

## 5. Split Training and Test Data Before Tuning

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])
print("Training features:", X_train.shape[1])
print("Test set is reserved for final evaluation.")

## 6. Baseline: Ordinary Linear Regression

In [ ]:
linear_model = make_pipeline(StandardScaler(), LinearRegression())
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

baseline_mse = mean_squared_error(y_test, linear_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_mae = mean_absolute_error(y_test, linear_pred)
baseline_r2 = r2_score(y_test, linear_pred)

print(f"Linear Regression MSE : {baseline_mse:.3f}")
print(f"Linear Regression RMSE: {baseline_rmse:.3f}")
print(f"Linear Regression MAE : {baseline_mae:.3f}")
print(f"Linear Regression R2  : {baseline_r2:.3f}")

This baseline is the reference for judging whether regularization improves held-out performance.

## 7. Candidate Alpha Grid

In [ ]:
alphas = np.logspace(-3, 3, 13)
print("Candidate alpha values:")
for alpha in alphas:
    print(f"{alpha:.6g}")

plt.figure(figsize=(8, 2.5))
plt.scatter(alphas, np.ones_like(alphas), s=65)
plt.xscale("log")
plt.yticks([])
plt.xlabel("Candidate alpha (log scale)")
plt.title("RidgeCV Alpha Search Grid")
plt.grid(True, axis="x")
plt.show()

A logarithmic grid explores weak through strong regularization. Expand the range if the selected alpha sits at either boundary.

## 8. Five-Fold Cross-Validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold_number, (fit_index, validation_index) in enumerate(cv.split(X_train), start=1):
    print(f"Fold {fold_number}: {len(fit_index)} fit rows, {len(validation_index)} validation rows")

K-fold CV repeatedly fits on four folds and validates on the remaining one. For time-ordered data, use time-aware splitting.

## 9. Fit RidgeCV with Scaling Inside the Pipeline

In [ ]:
ridge_model = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=alphas, cv=cv, scoring="neg_mean_squared_error")
)
ridge_model.fit(X_train, y_train)

ridgecv = ridge_model.named_steps["ridgecv"]
print("Selected alpha:", ridgecv.alpha_)
print("RidgeCV training completed.")

Negative MSE is used because scikit-learn maximizes scores; a less negative value means a lower MSE.

## 10. Evaluate RidgeCV on the Held-Out Test Set

In [ ]:
ridge_pred = ridge_model.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(ridge_mse)
ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_r2 = r2_score(y_test, ridge_pred)

print(f"Selected alpha: {ridgecv.alpha_:.6g}")
print(f"RidgeCV MSE : {ridge_mse:.3f}")
print(f"RidgeCV RMSE: {ridge_rmse:.3f}")
print(f"RidgeCV MAE : {ridge_mae:.3f}")
print(f"RidgeCV R2  : {ridge_r2:.3f}")

Lower MSE, RMSE and MAE mean smaller errors; higher R² means a better fit relative to a mean-only baseline.

## 11. Compare Linear Regression and RidgeCV

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "RidgeCV"],
    "MSE": [baseline_mse, ridge_mse],
    "RMSE": [baseline_rmse, ridge_rmse],
    "MAE": [baseline_mae, ridge_mae],
    "R2": [baseline_r2, ridge_r2]
})
display(comparison.round(3))

plt.figure(figsize=(7, 4))
sns.barplot(data=comparison, x="Model", y="RMSE")
plt.title("Held-Out Test RMSE Comparison")
plt.ylabel("RMSE (lower is better)")
plt.show()

Report the observed results; do not assume RidgeCV must outperform the baseline.

## 12. Examine Prediction Errors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, ridge_pred, alpha=0.7)
low = min(y_test.min(), ridge_pred.min())
high = max(y_test.max(), ridge_pred.max())
axes[0].plot([low, high], [low, high], linestyle="--")
axes[0].set_title("Actual vs Predicted: RidgeCV")
axes[0].set_xlabel("Actual values")
axes[0].set_ylabel("Predicted values")

residuals = y_test - ridge_pred
axes[1].scatter(ridge_pred, residuals, alpha=0.7)
axes[1].axhline(0, linestyle="--")
axes[1].set_title("Residuals vs Predicted")
axes[1].set_xlabel("Predicted values")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.show()

Look for systematic patterns, unusually large errors, and departures from a roughly random residual cloud.

## 13. Coefficient Shrinkage as Alpha Changes

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

coefficient_rows = []
for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    coefficient_rows.append(model.coef_)

coef_path = pd.DataFrame(coefficient_rows, columns=X.columns, index=alphas)

plt.figure(figsize=(11, 6))
for feature in coef_path.columns:
    plt.plot(coef_path.index, coef_path[feature], marker="o", label=feature)
plt.xscale("log")
plt.xlabel("Alpha (log scale)")
plt.ylabel("Standardized coefficient")
plt.title("Ridge Coefficient Shrinkage")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(True)
plt.tight_layout()
plt.show()

As alpha increases, coefficient magnitudes are pulled toward zero (the L2 effect).

## 14. Compare RidgeCV with Manual Cross-Validation

In [ ]:
cv_mse = []
for alpha in alphas:
    candidate = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    scores = cross_val_score(candidate, X_train, y_train, cv=cv,
                             scoring="neg_mean_squared_error")
    cv_mse.append(-scores.mean())

cv_results = pd.DataFrame({"alpha": alphas, "mean_cv_mse": cv_mse}).sort_values("mean_cv_mse")
display(cv_results)

plt.figure(figsize=(8, 4))
plt.plot(cv_results.sort_values("alpha")["alpha"],
         cv_results.sort_values("alpha")["mean_cv_mse"], marker="o")
plt.xscale("log")
plt.xlabel("Alpha (log scale)")
plt.ylabel("Mean cross-validation MSE")
plt.title("Cross-Validation Error Across Alpha Values")
plt.grid(True)
plt.show()

print("Best alpha from manual CV:", cv_results.iloc[0]["alpha"])

## 15. Practical Extension: Dataset with Multicollinearity

In [ ]:
from sklearn.datasets import make_regression

X_sim, y_sim = make_regression(n_samples=500, n_features=8, n_informative=6,
                               noise=25, random_state=42)
X_sim = pd.DataFrame(X_sim, columns=[f"x{i}" for i in range(8)])
X_sim["x8"] = X_sim["x0"] * 0.95 + np.random.default_rng(42).normal(0, 0.1, 500)
X_sim["x9"] = X_sim["x1"] * 0.90 + np.random.default_rng(7).normal(0, 0.1, 500)

print("Correlation between x0 and x8:", X_sim["x0"].corr(X_sim["x8"]))
print("Correlation between x1 and x9:", X_sim["x1"].corr(X_sim["x9"]))

## 16. Compare OLS and Ridge on Correlated Predictors

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X_sim, y_sim, test_size=0.2, random_state=42)

ols = make_pipeline(StandardScaler(), LinearRegression())
ridge = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=np.logspace(-3, 3, 25),
            cv=KFold(n_splits=5, shuffle=True, random_state=42),
            scoring="neg_mean_squared_error")
)
ols.fit(Xtr, ytr)
ridge.fit(Xtr, ytr)

for name, estimator in [("OLS", ols), ("RidgeCV", ridge)]:
    pred = estimator.predict(Xte)
    print(name, "| RMSE:", round(np.sqrt(mean_squared_error(yte, pred)), 3),
          "| R2:", round(r2_score(yte, pred), 3))
print("Ridge selected alpha:", ridge.named_steps["ridgecv"].alpha_)

Discuss both predictive performance and coefficient stability; lower test error is not guaranteed in every sample.

## 17. Optional: Use Your Own CSV Dataset in Colab
Set `USE_CUSTOM_CSV = True`, run the cell, upload your file, and change `TARGET` to your target column name.

In [ ]:
USE_CUSTOM_CSV = False   # set True to use your own data
TARGET = "target"        # CHANGE to your target column

if USE_CUSTOM_CSV:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
    custom_df = pd.read_csv(file_name)
    display(custom_df.head())
    print(custom_df.shape)
    print(custom_df.dtypes)
    display(custom_df.describe(include="all").T)

    X_custom = custom_df.drop(columns=[TARGET])
    y_custom = custom_df[TARGET]
    X_custom = X_custom.select_dtypes(include=np.number)
    print("Numeric predictors:", X_custom.columns.tolist())
else:
    print("Skipped (USE_CUSTOM_CSV = False).")

For categorical variables or missing values, add suitable preprocessing inside a scikit-learn Pipeline before fitting RidgeCV.

## 18. End-to-End Template for a Numeric CSV Dataset

In [ ]:
if USE_CUSTOM_CSV:
    Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(
        X_custom, y_custom, test_size=0.20, random_state=42)

    custom_model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=np.logspace(-3, 3, 25),
                cv=KFold(n_splits=5, shuffle=True, random_state=42),
                scoring="neg_mean_squared_error")
    )
    custom_model.fit(Xtr_c, ytr_c)
    custom_pred = custom_model.predict(Xte_c)

    print("Selected alpha:", custom_model.named_steps["ridgecv"].alpha_)
    print("Test RMSE:", np.sqrt(mean_squared_error(yte_c, custom_pred)))
    print("Test MAE:", mean_absolute_error(yte_c, custom_pred))
    print("Test R2:", r2_score(yte_c, custom_pred))
else:
    print("Skipped (USE_CUSTOM_CSV = False).")

---
# Practical Assignment and Reporting
The sections below cover the assignment requirements. Write your interpretations in the markdown cells.

## A1. Alpha tuning with a narrow grid and a wider logarithmic grid

In [ ]:
grids = {
    "Narrow grid (0.1 to 10, 9 values)": np.logspace(-1, 1, 9),
    "Wide grid (1e-4 to 1e4, 33 values)": np.logspace(-4, 4, 33),
}

rows = []
for name, grid in grids.items():
    m = make_pipeline(StandardScaler(),
                      RidgeCV(alphas=grid, cv=cv, scoring="neg_mean_squared_error"))
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    a = m.named_steps["ridgecv"].alpha_
    rows.append({
        "Grid": name,
        "Selected alpha": a,
        "At boundary?": bool(np.isclose(a, grid.min()) or np.isclose(a, grid.max())),
        "Test MSE": mean_squared_error(y_test, p),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, p)),
        "Test MAE": mean_absolute_error(y_test, p),
        "Test R2": r2_score(y_test, p),
    })

grid_results = pd.DataFrame(rows)
display(grid_results.round(4))

**Interpretation (write here):** Which alpha did each grid select? Did the narrow grid hit a boundary? Did the wider grid change test performance?

## A2. Linear Regression vs RidgeCV (all four metrics)

In [ ]:
final = pd.concat([
    comparison.set_index("Model"),
    grid_results.set_index("Grid")[["Test MSE","Test RMSE","Test MAE","Test R2"]]
    .rename(columns=lambda c: c.replace("Test ", ""))
]).round(3)
display(final)

**Interpretation (write here):** Does RidgeCV beat OLS on the held-out set? By how much? Is the difference practically meaningful?

## A3. Correlation heatmap: identify highly correlated predictors

In [ ]:
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Predictor-Predictor Correlation (lower triangle)")
plt.show()

pairs = (corr.where(~mask).stack().rename("correlation").reset_index()
         .rename(columns={"level_0": "feature_1", "level_1": "feature_2"}))
pairs["abs_corr"] = pairs["correlation"].abs()
high_pairs = pairs[pairs["abs_corr"] >= 0.5].sort_values("abs_corr", ascending=False)
print("Predictor pairs with |r| >= 0.5:")
display(high_pairs.drop(columns="abs_corr").round(3))

**Interpretation (write here):** Name the strongly correlated predictors and explain why this causes unstable OLS coefficients.

## A4. Coefficient shrinkage: what happens as alpha increases

In [ ]:
ols_coefs = pd.Series(LinearRegression().fit(X_train_scaled, y_train).coef_, index=X.columns)
coef_compare = pd.DataFrame({
    "OLS": ols_coefs,
    "Ridge alpha=0.001": coef_path.iloc[0],
    f"Ridge alpha={ridgecv.alpha_:.3g} (CV)": pd.Series(
        ridge_model.named_steps["ridgecv"].coef_, index=X.columns),
    "Ridge alpha=1000": coef_path.iloc[-1],
})
display(coef_compare.round(2))
print("Sum of |coef|:")
display(coef_compare.abs().sum().round(1).to_frame("L1 norm"))

**Interpretation (write here):** At very small alpha, Ridge matches OLS; as alpha grows, coefficients shrink toward zero but are never exactly zero. Correlated predictors (e.g. s1 and s2) with large opposite-signed OLS coefficients are shrunk the most.

---
# Novelty and Contribution (extension of the main reference)
The extensions below go beyond the base practical. Edit, extend, or replace them with your own idea, and state clearly what you added.

1. **VIF analysis** to quantify multicollinearity per predictor.
2. **Repeated 5-fold CV** for a more stable comparison of OLS, RidgeCV, LassoCV and ElasticNetCV.
3. **Bootstrap coefficient stability**: how much do OLS vs Ridge coefficients vary across resamples?
4. **One-standard-error rule** for a more conservative alpha.

### N1. Variance Inflation Factors

In [ ]:
def vif_table(features):
    R = np.corrcoef(features.values, rowvar=False)
    return pd.Series(np.diag(np.linalg.inv(R)), index=features.columns, name="VIF")

vif_diabetes = vif_table(X)
vif_sim = vif_table(X_sim)
display(pd.concat([vif_diabetes.sort_values(ascending=False)], axis=1).round(2))
display(vif_sim.sort_values(ascending=False).to_frame().round(2))
print("Rule of thumb: VIF > 5 (or 10) signals problematic multicollinearity.")

### N2. Repeated CV: OLS vs Ridge vs Lasso vs ElasticNet

In [ ]:
from sklearn.model_selection import RepeatedKFold
from sklearn.linear_model import LassoCV, ElasticNetCV

rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
models = {
    "OLS": make_pipeline(StandardScaler(), LinearRegression()),
    "RidgeCV": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25))),
    "LassoCV": make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=20000)),
    "ElasticNetCV": make_pipeline(StandardScaler(),
                                  ElasticNetCV(l1_ratio=[.1, .5, .9], cv=5,
                                               random_state=RANDOM_STATE, max_iter=20000)),
}

rows = []
for name, mdl in models.items():
    sc = -cross_val_score(mdl, X, y, cv=rkf, scoring="neg_root_mean_squared_error")
    rows.append({"Model": name, "Mean CV RMSE": sc.mean(), "Std": sc.std()})
rep_results = pd.DataFrame(rows).sort_values("Mean CV RMSE")
display(rep_results.round(3))

plt.figure(figsize=(7, 4))
plt.bar(rep_results["Model"], rep_results["Mean CV RMSE"], yerr=rep_results["Std"], capsize=5)
plt.ylabel("Mean CV RMSE (lower is better)")
plt.title("Repeated 5x5 CV RMSE (diabetes data, all rows)")
plt.show()

### N3. Bootstrap coefficient stability (OLS vs Ridge)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
best_alpha = ridgecv.alpha_
n = len(X_train)
ols_boot, ridge_boot = [], []

for _ in range(200):
    idx = rng.integers(0, n, n)
    Xb, yb = X_train.iloc[idx], y_train.iloc[idx]
    ols_boot.append(make_pipeline(StandardScaler(), LinearRegression()).fit(Xb, yb)
                    .named_steps["linearregression"].coef_)
    ridge_boot.append(make_pipeline(StandardScaler(), Ridge(alpha=best_alpha)).fit(Xb, yb)
                      .named_steps["ridge"].coef_)

stab = pd.DataFrame({
    "OLS coef std": np.std(ols_boot, axis=0),
    "Ridge coef std": np.std(ridge_boot, axis=0),
}, index=X.columns)
stab["Variance reduction %"] = 100 * (1 - stab["Ridge coef std"] / stab["OLS coef std"])
display(stab.round(2))

stab[["OLS coef std", "Ridge coef std"]].plot(kind="bar", figsize=(9, 4))
plt.title("Bootstrap Coefficient Variability (200 resamples)")
plt.ylabel("Std of standardized coefficient")
plt.tight_layout()
plt.show()

**Interpretation (write here):** Which coefficients become most stable under Ridge? Do they match the high-VIF predictors from N1?

### N4. One-standard-error rule for alpha

In [ ]:
fine_alphas = np.logspace(-3, 3, 41)
fold_mse = np.array([
    -cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=a)),
                     X_train, y_train, cv=cv, scoring="neg_mean_squared_error")
    for a in fine_alphas
])
mean_mse = fold_mse.mean(axis=1)
se_mse = fold_mse.std(axis=1, ddof=1) / np.sqrt(fold_mse.shape[1])

best_i = mean_mse.argmin()
threshold = mean_mse[best_i] + se_mse[best_i]
alpha_1se = fine_alphas[np.where(mean_mse <= threshold)[0].max()]

print(f"Alpha with minimum CV MSE : {fine_alphas[best_i]:.4g}")
print(f"Alpha by 1-SE rule        : {alpha_1se:.4g}")

plt.figure(figsize=(8, 4))
plt.errorbar(fine_alphas, mean_mse, yerr=se_mse, fmt="-o", ms=3)
plt.axvline(fine_alphas[best_i], ls="--", label="min CV MSE")
plt.axvline(alpha_1se, ls=":", color="red", label="1-SE rule")
plt.xscale("log"); plt.xlabel("Alpha (log scale)"); plt.ylabel("CV MSE")
plt.title("CV Error with Standard Errors"); plt.legend(); plt.grid(True)
plt.show()

m1se = make_pipeline(StandardScaler(), Ridge(alpha=alpha_1se)).fit(X_train, y_train)
print("Test RMSE with 1-SE alpha:", round(np.sqrt(mean_squared_error(y_test, m1se.predict(X_test))), 3))
print("Test RMSE with min-CV alpha:", round(ridge_rmse, 3))

## Conclusions (write your own)
- **Main result:** ...
- **Effect of the alpha grid (narrow vs wide):** ...
- **Multicollinearity findings (heatmap, VIF):** ...
- **What shrinkage did to the coefficients:** ...
- **My novelty / contribution:** ...
- **Limitations:** single train/test split, small dataset, linear model only.